# Deep Learning Models Training(CNN,CRNN)

This Notebook is used to train deep learning models to complete Music Genre Classification.
The training model includes:

-CNN (Convolutional Neural Network)
-CRNN (Convolutional Recurrent Neural Network)

The training data uses Mel Spectrogram obtained by preprocessing in advance, and evaluates the model performance in combination with 5-fold Cross Validation.
Finally, use the entire training set to retrain the best model and save the trained parameters for subsequent testing and model evaluation.

## 1 Import Libaraies
The following libraries are imported fordata processing,deep learning,cross validation and model evaluation.

In [3]:
import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset, Subset

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

from model import CNN, CRNN
from train import GTZANDdataset,cross_validation,BATCH_SIZE,LEARNING_RATE,EPOCHS,NUM_CLASSES,DEVICE,Mel_dir


## 2 Prepare Training Dataset
This part is intended to prepare the data which will be used for training the models.
Workflow:X_train.csv-->Read filenames-->y_train.csv-->Read corresponding genre labels-->Locate Mel Spectrogram (.npy)
      -->Load Mel Spectrogram-->Standardize-->Convert to Tensor
(Dataset preparation is implemented in train.py.This notebook directly imports GTZANDdataset.)

## 3 Training Configuration

Batch Size：The number of samples used for each update of parameters
    Batch Size = 32

Learning Rate：Adam Optimizer learning rate
    Learning Rate = 0.0005

Epoch：The number of training sets that the model has been completed
    Epoch = 50

Number of Classes：Genres in GTZAN dataset：10 Genres。
    Number of Classes = 10

Device：Autometically detect GPU
    if GPU is available：CUDA
    otherwise：CPU

In [4]:
from train import (
    BATCH_SIZE,
    LEARNING_RATE,
    EPOCHS,
    NUM_CLASSES,
    DEVICE,
    Mel_dir,
)

print(f"Batch Size: {BATCH_SIZE}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Epochs: {EPOCHS}")
print(f"Device: {DEVICE}")

Batch Size: 32
Learning Rate: 0.0005
Epochs: 50
Device: cpu


## 4 Load Traing Data
Load the pre-generated training feature and label files, then construct the PyTorch dataset for model training.

In [5]:
# Load training CSV files
x_train = pd.read_csv("X_train.csv")
y_train = pd.read_csv("y_train.csv")

# Create dataset
train_dataset = GTZANDdataset(Mel_dir, x_train, y_train)

print("Dataset Information")
print("-" * 40)
print(f"Training samples : {len(train_dataset)}")

labels = [label for _, label in train_dataset.samples]
print(f"Number of classes: {len(set(labels))}")

sample, label = train_dataset[0]
print(f"Input shape      : {tuple(sample.shape)}")
print(f"Sample label     : {label.item()}")

Dataset Information
----------------------------------------
Training samples : 804
Number of classes: 10
Input shape      : (1, 128, 1292)
Sample label     : 0


## 5 Cross Validation Setup
In order to improve the stability and classification performance of model training,our project adopts：5-Fold Cross Validation
Workflow：Training Dataset-->Split into 5 Folds-->4 Folds Training-->1 Fold Validation-->Repeat 5 Times
The macro F1 score is used as an evaluation indicator because it gives equal attention to each music genre.

In [6]:
kf = StratifiedKFold(
    n_splits = 5, 
    shuffle = True, 
    random_state = 42
)
print("Cross Validation Configuration")
print("- Number of folds : 5")
print("- Split method    : StratifiedKFold")
print("- Shuffle         : True")
print("- Random state    : 42")

Cross Validation Configuration
- Number of folds : 5
- Split method    : StratifiedKFold
- Shuffle         : True
- Random state    : 42


## 6 Training Pipeline
This part introduces the training process of CNN and CRNN. CNN and CRNN use the same training process. ( In order to better compare the differences between CNN and CRNN in dealing with music genres)
The training data adopts the preprocessed Mel Spectrogram, and the 5-Fold Cross Validation is used to evaluate the model performance. During the training process, the model learns the characteristic representation of different music genres by constantly updating the parameters, so as to improve the classification accuracy.

- Adam Optimizer is adopted to update network parameters because it combines momentum and adaptive learning rates, leading to faster and more stable convergence.

- Cross Entropy Loss is used for multi-class classification by measuring the difference between predicted probabilities and the true labels.
Five-fold Cross Validation evaluates the model's generalization performance. The average Macro F1 Score across all folds is used as the model evaluation criterion.

- Five-fold Cross Validation evaluates the model's generalization performance. The average Macro F1 Score across all folds is used as the model evaluation criterion.

## 7 CNN training

 ### 7.1 CNN training process
The CNN model is trained using five-fold stratified cross validation.  
- For each fold, the model is trained for 50 epochs using the Adam optimizer and CrossEntropyLoss.  
- The best Macro F1 score on the validation set is recorded for each fold, and the average Macro F1 across all folds is used to evaluate the model.

In [7]:
print("=" * 60)
print("CNN Training")
print("=" * 60)

cnn_avg_f1, cnn_fold_results = cross_validation(
    train_dataset,
    CNN
)

CNN Training
Fold 1
Epoch [1/50] Loss: 2.3308 Train Accuracy: 14.15% Validation Accuracy: 14.91% Macro F1:0.0808
Epoch [2/50] Loss: 2.2168 Train Accuracy: 18.97% Validation Accuracy: 21.12% Macro F1:0.1228
Epoch [3/50] Loss: 2.1629 Train Accuracy: 20.06% Validation Accuracy: 31.68% Macro F1:0.2338
Epoch [4/50] Loss: 2.0624 Train Accuracy: 23.33% Validation Accuracy: 36.02% Macro F1:0.2777
Epoch [5/50] Loss: 2.0263 Train Accuracy: 26.91% Validation Accuracy: 36.65% Macro F1:0.2560
Epoch [6/50] Loss: 1.9998 Train Accuracy: 26.44% Validation Accuracy: 36.65% Macro F1:0.2729
Epoch [7/50] Loss: 2.0159 Train Accuracy: 29.86% Validation Accuracy: 39.13% Macro F1:0.2958
Epoch [8/50] Loss: 1.9606 Train Accuracy: 30.64% Validation Accuracy: 39.13% Macro F1:0.2980
Epoch [9/50] Loss: 1.9566 Train Accuracy: 30.02% Validation Accuracy: 35.40% Macro F1:0.2532
Epoch [10/50] Loss: 1.9132 Train Accuracy: 30.95% Validation Accuracy: 40.37% Macro F1:0.3027
Epoch [11/50] Loss: 1.8948 Train Accuracy: 33.28%

### 7.2 Training results
The following table summarises the best Macro F1 score achieved in each fold during five-fold cross validation. The average Macro F1 is reported as the overall training performance of the CNN model.

In [8]:
cnn_results_df = pd.DataFrame(cnn_fold_results)

print("CNN Cross Validation Results")
display(cnn_results_df)

print(f"\nAverage CNN Macro F1: {cnn_avg_f1:.4f}")

CNN Cross Validation Results


,Model,Fold,Macro F1
0,CNN,1,0.4597
1,CNN,2,0.4796
2,CNN,3,0.4953
3,CNN,4,0.4656
4,CNN,5,0.4710



Average CNN Macro F1: 0.4742


## 8 CRNN training

### 8.1 CRNN training process
The CRNN model follows the same five-fold stratified cross validation procedure as the CNN model. The convolutional layers first extract spatial features from Mel spectrograms, and the LSTM captures temporal dependencies before classification.

In [9]:
print("=" * 60)
print("CRNN Training")
print("=" * 60)

crnn_avg_f1, crnn_fold_results = cross_validation(
    train_dataset,
    CRNN
)

CRNN Training
Fold 1
Epoch [1/50] Loss: 2.2808 Train Accuracy: 17.88% Validation Accuracy: 25.47% Macro F1:0.1470
Epoch [2/50] Loss: 2.0717 Train Accuracy: 30.95% Validation Accuracy: 27.33% Macro F1:0.1692
Epoch [3/50] Loss: 1.9211 Train Accuracy: 31.26% Validation Accuracy: 31.06% Macro F1:0.2255
Epoch [4/50] Loss: 1.8166 Train Accuracy: 34.84% Validation Accuracy: 38.51% Macro F1:0.3275
Epoch [5/50] Loss: 1.7292 Train Accuracy: 38.41% Validation Accuracy: 35.40% Macro F1:0.2736
Epoch [6/50] Loss: 1.6985 Train Accuracy: 41.68% Validation Accuracy: 36.02% Macro F1:0.2788
Epoch [7/50] Loss: 1.7674 Train Accuracy: 35.61% Validation Accuracy: 26.09% Macro F1:0.1702
Epoch [8/50] Loss: 1.5597 Train Accuracy: 39.97% Validation Accuracy: 39.75% Macro F1:0.3000
Epoch [9/50] Loss: 1.5851 Train Accuracy: 39.66% Validation Accuracy: 43.48% Macro F1:0.3474
Epoch [10/50] Loss: 1.6816 Train Accuracy: 38.57% Validation Accuracy: 37.27% Macro F1:0.2931
Epoch [11/50] Loss: 1.9235 Train Accuracy: 30.95

### 8.2 CRNN training results
The best Macro F1 score from each fold is summarised below. The average Macro F1 is used as the overall performance indicator of the CRNN model.

In [10]:
crnn_results_df = pd.DataFrame(crnn_fold_results)

print("CRNN Cross Validation Results")
display(crnn_results_df)

print(f"\nAverage CRNN Macro F1: {crnn_avg_f1:.4f}")

CRNN Cross Validation Results


,Model,Fold,Macro F1
0,CRNN,1,0.5552
1,CRNN,2,0.5818
2,CRNN,3,0.5103
3,CRNN,4,0.5738
4,CRNN,5,0.5591



Average CRNN Macro F1: 0.5560
